<a href="https://colab.research.google.com/github/chromeburner/SJPL-Webcrawler/blob/main/Python_Webscraper_AI_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

BASE_URL = "https://sjpl.bibliocommons.com/v2/events?locations=25&endDate=2025-09-30"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; EventScraper/1.0)"
}

def fetch_events_page(page_number):
    # Correctly append the page number as a query parameter
    url = f"{BASE_URL}&page={page_number}"
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    return response.text

def parse_events(html):
    soup = BeautifulSoup(html, "html.parser")

    titles = soup.find_all("div", class_="cp-event-title")
    dates = soup.find_all("div", class_="cp-event-date-time")
    locations = soup.find_all("div", class_="cp-event-location-name")
    descriptions = soup.find_all("div", class_="cp-event-description")

    total = min(len(titles), len(dates), len(locations), len(descriptions))
    events = []
    for i in range(total):
        title = titles[i].get_text(strip=True)

        # Clean and format the date and time string
        date_time_text = dates[i].get_text(strip=True)
        # Split the string by "on" to separate the day/date from the time information
        date_parts = date_time_text.split("on")
        if len(date_parts) > 1:
            # The first part is the day and date (e.g., "Friday, September 05")
            day_and_date = date_parts[0].strip()
            # The second part contains the date and time ranges (e.g., "September 5, 2025,3:30pm–4:30pm3:30pm to 4:30pm")
            time_parts = date_parts[1].strip().split(',')
            if len(time_parts) > 2:
                 # The time range is usually after the second comma (e.g., "3:30pm–4:30pm")
                time_range = time_parts[2].strip().split('m')[0] + 'm–' + time_parts[2].strip().split('m')[2] + 'm'
            else:
                # Handle cases where the format might be slightly different
                time_range = ""
            cleaned_date_time = f"{day_and_date}, {time_range}"
        else:
            # If "on" is not in the string, use the original text
            cleaned_date_time = date_time_text


        location_text = locations[i].get_text(strip=True)
        # Clean and format the location string
        if "Event location:" in location_text:
            location = "Event Location: " + location_text.split("Event location:")[1].strip()
        else:
            location = "Event Location: " + location_text.strip()

        description = descriptions[i].get_text(strip=True)
        events.append((title, cleaned_date_time, location, description))

    return events

def main():
    page = 1
    all_events = []

    while True:
        print(f"Fetching page {page}...")
        html = fetch_events_page(page)
        events = parse_events(html)

        if not events:
            print("No more events found. Stopping.")
            break

        all_events.extend(events)
        page += 1

        time.sleep(1)  # be polite and avoid hammering the server

    # Print all events
    for title, date_time, location, description in all_events:
        print(title)
        print(date_time)
        print(location)
        print(description)
        print()  # blank line between events

if __name__ == "__main__":
    main()

Fetching page 1...
Fetching page 2...
Fetching page 3...
No more events found. Stopping.
STEAM Club for Kids
Friday, September 05, 3:30pm–3:30pm
Event Location: Village Square
The STEAM Club for Kids is a monthly, one-hour program designed to engage students in hands-on learning across science, technology, engineering, arts, and mathematics. Each session introduces a unique STEAM theme featuring an interactive…

Village Square Chess Club
Saturday, September 06, 10:00am–10:00am
Event Location: Village Square
Are you interested in playing and learning chess? If so, you are invited at the Village Square Branch Library to come and learn to play chess at the beginners and intermediate levels. Walk-ins are welcome!Our Village Square Chess Club…

Friends of Village Square Book Club
Saturday, September 06, 10:30am–10:30am
Event Location: Village Square
Read our fiction or non-fiction book selection for the month and meet some new friends! Our monthly book club typically meets on the first Satu